In [10]:
import os
import sys
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from model import Unet
import cv2
from utils import root, dice_score, save_checkpoint, load_checkpoint
import matplotlib.pyplot as plt
import numpy as np

sys.path.append(root)
from dataset.dataloader import get_data_loaders

In [11]:
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
NUM_EPOCHS = 25
NUM_WORKERS = 2
MAX_DIM = 512
PIN_MEMORY = True
LOAD_MODEL = False

In [3]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)
    
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE)
        targets = targets.float().to(device=DEVICE)
        
        with torch.cuda.amp.autocast():
            predictions = model(data)
            print(predictions.shape)
            print(targets.shape)
            loss = loss_fn(predictions, targets)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loop.set_postfix(loss=loss.item())

In [4]:
transform = A.Compose(
        [
            A.LongestMaxSize(max_size=MAX_DIM, always_apply=True),
            A.PadIfNeeded(min_height=MAX_DIM, min_width=MAX_DIM, border_mode=cv2.BORDER_CONSTANT, value=[0, 0, 0], always_apply=True),
            A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0),
            ToTensorV2(),
        ]
    )
    
model = Unet().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_ldr, valid_ldr, test_ldr = get_data_loaders(transform, transform, transform, 16, 0.1, 0.1)

In [9]:
def validate_fn(valid_loader, model):
    model.eval()
    dice_scores = []
    with torch.no_grad():
        for images, targets in valid_loader:
            images = images.cuda()
            targets = targets.cuda()
            outputs = model(images)
            predictions = torch.sigmoid(outputs) > 0.5
            dice = dice_score(predictions, targets)
            dice_scores.append(dice.item())
    return sum(dice_scores) / len(dice_scores)

In [12]:
scaler = torch.cuda.amp.GradScaler()
for epoch in range(NUM_EPOCHS):
    train_fn(train_ldr, model, optimizer, loss_fn, scaler)
    mean_dice_score = validate_fn(valid_ldr, model)
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Mean Dice Score: {mean_dice_score}')
    checkpoint = {
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    save_dir = os.path.join(root, "unet", "checkpoints")
    name = f"unet_{epoch}"
    save_checkpoint(checkpoint, save_dir, name)

  0%|                                                                                            | 0/9 [00:32<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB (GPU 0; 4.00 GiB total capacity; 6.37 GiB already allocated; 0 bytes free; 6.54 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF